# MOTEL workflow diagram

This diagram shows the current MOTEL workflow across two connected repositories. It distinguishes data artifacts, process steps, supporting context, and user-facing outputs while showing how `motel-platform` feeds a separate ontology, knowledge-graph, and web-app stack.

```mermaid
flowchart TB
    classDef data fill:#eaf3ff,stroke:#4c77a8,stroke-width:2px,color:#14324f;
    classDef process fill:#e9f7f0,stroke:#2f7f5b,stroke-width:2px,color:#123326;
    classDef context fill:#fff6de,stroke:#b4871f,stroke-width:2px,color:#5e450a;
    classDef consumer fill:#f4ecff,stroke:#7a58b5,stroke-width:2px,color:#3b245f;
    classDef repo fill:#f8fbfa,stroke:#c8ddd2,stroke-width:1px,color:#355348;

    subgraph repo_platform[Current repo: motel-platform]
        src[(Raw source data\nspreadsheets, reports, manual review)]:::data --> ingest[Ingestion process\nnotebook + helper module]:::process
        schemas[[Schema context\nschema + schema_simple]]:::context --> ingest
        ingest --> unmapped[(Unmapped staging data\nmotel-db/unmapped_entity YAML)]:::data
        unmapped --> harmonise[Harmonisation process\n2_harmonise notebook + helpers]:::process
        existing[[Existing MOTEL registries\nlookup, provenance, review context]]:::context --> harmonise
        harmonise --> registries[(Controlled vocabularies + secondary registries\nmotel-db/controlled_vocabulary and secondary)]:::data
        harmonise --> mappings[(Mapping tables\nmotel-db/mapping)]:::data
        harmonise --> linked[(Linked entity outputs\nlinked structure and harmonised records)]:::data
        registries --> release([Repository release + docs]):::consumer
        mappings --> release
        linked --> release
    end

    subgraph repo_kg[Separate repo: MOTEL ontology, KG, and web app]
        ontology[[Ontology definitions\ntechnologies, attributes, flows]]:::context --> ttl_process[TTL conversion + graph preparation]:::process
        ttl_process --> ttl[(Ontology-based TTL files)]:::data
        ttl --> graphdb[(GraphDB repository)]:::data
        graphdb --> api[FastAPI backend]:::process
        api --> webapp([Next.js frontend\nexploration and demo app]):::consumer
    end

    registries --> ttl_process
    mappings --> ttl_process
    linked --> ttl_process
    release --> users([Modelers, reviewers, ORD users]):::consumer
    webapp --> users
```

## Read this diagram as

- The left subgraph is the current `motel-platform` repository: ingestion, harmonisation, and publication of structured MOTEL data.
- The right subgraph is a separate downstream repository: ontology-based TTL generation, GraphDB loading, backend API serving, and frontend exploration.
- Blue nodes are data artifacts, green nodes are process steps, yellow nodes are supporting context, and purple nodes are user-facing outputs or consumers.
- The connection between the two repositories is the structured MOTEL output: registries, mappings, and linked records feed the ontology and graph stack.
